# Binary PTM Text Classifier: Light vs Dark Kinase

**Goal:** Given a PTM comment text from UniProt, predict whether the protein is a
*light* kinase (well-characterized, in PhosphoAtlas) or a *dark* kinase (understudied).

**Why this is hard:** The label comes from experimental pull-down data, not the text itself.
The signal we're capturing is *research attention* — dark kinase PTM entries tend to be
shorter, vaguer, and lack named kinases/residues, simply because less work has been done on them.

**Three models compared:**

| Model | Approach | Training required? |
|-------|----------|--------------------|
| 1. Baseline | Frozen PubMedBERT + Logistic Regression | No (embeddings only) |
| 2. Fine-tuned | PubMedBERT fully fine-tuned | Yes |
| 3. Zero-shot | GPT-4.1 with a crafted prompt | No |

In [ ]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from torch.nn import CrossEntropyLoss
from transformers import (
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

sys.path.insert(0, "..")
from phospho.llm_normalize import get_llm_client

warnings.filterwarnings("ignore")
%config InlineBackend.figure_format = 'retina'

# Shared constants
MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
RANDOM_STATE = 42
MAX_LENGTH = 128   # PTM texts are short (~20-60 tokens); 128 is sufficient and faster than 512
CLASS_NAMES = ["light", "dark"]

## Dataset

Source: `data/updated_PTM_dataset_03_10_2026.csv`

**Label remapping:**
- `output_class 0` (PhosphoAtlas background) → **label 0 (light)**
- `output_class 1` (dark kinase bait) → **label 1 (dark)**
- `output_class 2` (dark kinase prey) → **label 1 (dark)**

We collapse bait and prey into one dark class because both represent proteins
found in the understudied dark kinome — the distinction only matters for the
AP-MS experimental context, not for text-based classification.

> **⚠️ Class imbalance:** ~87% dark, ~13% light. A naive model that predicts
> 'dark' for everything would get 87% accuracy but be useless.
> This is why we use **F1-macro** as our primary metric — it penalizes
> ignoring the minority class equally.

In [ ]:
df = pd.read_csv("../data/updated_PTM_dataset_03_10_2026.csv")

# Remap to binary labels
df["label"] = df["output_class"].apply(lambda x: 0 if x == 0 else 1)

print(f"Total samples: {len(df)}")
print(f"\nClass distribution:")
for i, name in enumerate(CLASS_NAMES):
    count = (df["label"] == i).sum()
    pct = count / len(df) * 100
    print(f"  {i} — {name}: {count} ({pct:.1f}%)")

print(f"\nSample PTM texts:")
for label, name in enumerate(CLASS_NAMES):
    sample = df[df["label"] == label]["ptm_text"].iloc[0]
    print(f"\n[{name}] {sample[:120]}...")

In [ ]:
# Stratified 80/10/10 split — stratify ensures both classes appear in each split
# at the same ratio as the full dataset (critical when classes are imbalanced)
model_df = df[["ptm_text", "label"]].copy()

train_val_df, test_df = train_test_split(
    model_df, test_size=0.10, random_state=RANDOM_STATE, stratify=model_df["label"]
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.111, random_state=RANDOM_STATE, stratify=train_val_df["label"]
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"\nTrain class distribution:")
for i, name in enumerate(CLASS_NAMES):
    count = (train_df["label"] == i).sum()
    print(f"  {name}: {count}")

---
## Model 1: Baseline — Frozen PubMedBERT + Logistic Regression

**What we're doing:** Run every PTM text through PubMedBERT *without changing any of its weights*.
Extract the `[CLS]` token embedding (a 768-dimensional vector that summarizes the whole sentence),
then train a simple logistic regression on top of those vectors.

**Why PubMedBERT specifically?**
Regular BERT was trained on Wikipedia and books. PubMedBERT was trained from scratch on PubMed
abstracts and full-text articles — so it already understands words like "phosphorylation",
"Thr-308", and "MAPK" as meaningful tokens, not rare noise.

**What the `[CLS]` token is:**
BERT adds a special `[CLS]` token at the start of every input. By the end of the model's
12 attention layers, this token has aggregated contextual information from the entire sequence.
It's the standard representation to use for classification.

**What this baseline tells us:**
Without any task-specific training, do PubMedBERT's pre-trained representations already
separate light from dark kinase descriptions? If yes, the signal is strong. If no, we need fine-tuning.

> **🎙️ Interview flag:** *Why logistic regression and not a neural head?*
> Because the embeddings are fixed — there's nothing deep to learn from them.
> Logistic regression is fast, interpretable, and hard to overfit.
> A deeper head would only help if we were also updating the embedding weights.

In [ ]:
# Load PubMedBERT as a base encoder (no classification head)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_encoder = AutoModel.from_pretrained(MODEL_NAME)

# Freeze all weights — we are using this purely as a feature extractor
for param in bert_encoder.parameters():
    param.requires_grad = False

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
bert_encoder = bert_encoder.to(device)
bert_encoder.eval()

print(f"Encoder loaded on: {device}")
print(f"Trainable parameters: {sum(p.numel() for p in bert_encoder.parameters() if p.requires_grad):,}  (frozen — none)")


def extract_cls_embeddings(texts: list[str], batch_size: int = 32) -> np.ndarray:
    """Pass texts through frozen BERT and return [CLS] embeddings.
    
    We process in batches to avoid loading all texts into GPU memory at once.
    Output shape: (n_texts, 768)
    """
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(
            batch,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            outputs = bert_encoder(**inputs)

        # last_hidden_state shape: (batch, seq_len, 768)
        # index 0 on seq_len dim = [CLS] token
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)

    return np.vstack(all_embeddings)


print("\nExtracting embeddings for train/val/test...")
X_train = extract_cls_embeddings(train_df["ptm_text"].tolist())
X_val   = extract_cls_embeddings(val_df["ptm_text"].tolist())
X_test  = extract_cls_embeddings(test_df["ptm_text"].tolist())

y_train = train_df["label"].values
y_val   = val_df["label"].values
y_test  = test_df["label"].values

print(f"Embedding shape: {X_train.shape}  → {X_train.shape[1]}-dim vector per text")

In [ ]:
# class_weight='balanced' automatically sets weights inversely proportional to class frequency
# Without this, logistic regression would heavily favor predicting 'dark' (87% of data)
#
# KEY PARAMETER: C=1.0 is the inverse regularization strength
#   - lower C = stronger regularization (simpler model, less overfitting)
#   - higher C = weaker regularization (fits training data more closely)
baseline_clf = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    C=1.0,
    random_state=RANDOM_STATE,
)
baseline_clf.fit(X_train, y_train)

# Evaluate on validation set
val_preds = baseline_clf.predict(X_val)
print("=== Model 1: Baseline — Validation Set ===")
print(classification_report(y_val, val_preds, target_names=CLASS_NAMES))

In [ ]:
# Final evaluation on held-out test set
baseline_preds = baseline_clf.predict(X_test)

print("=== Model 1: Baseline — Test Set ===")
print(classification_report(y_test, baseline_preds, target_names=CLASS_NAMES))

# Confusion matrix
cm1 = confusion_matrix(y_test, baseline_preds)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm1, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(CLASS_NAMES); ax.set_yticklabels(CLASS_NAMES)
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm1[i, j], ha="center", va="center", fontsize=14)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Model 1: Baseline — Confusion Matrix")
plt.tight_layout()
plt.show()

# Store results for final comparison
baseline_results = {
    "model": "1. Baseline (Frozen BERT + LogReg)",
    "accuracy":  accuracy_score(y_test, baseline_preds),
    "f1_macro":  f1_score(y_test, baseline_preds, average="macro"),
    "precision": precision_score(y_test, baseline_preds, average="macro"),
    "recall":    recall_score(y_test, baseline_preds, average="macro"),
}

---
## Model 2: Fine-tuned PubMedBERT

**What we're doing:** Now we *update* PubMedBERT's weights on our task. A new
binary classification head is added on top of the `[CLS]` embedding, and the
whole model is trained end-to-end.

**Key parameters to know:**

| Parameter | Value | Why |
|-----------|-------|-----|
| `learning_rate` | 2e-5 | Standard for BERT fine-tuning. Too high (e.g. 1e-4) destroys pre-trained weights ("catastrophic forgetting") |
| `num_train_epochs` | 5 | Enough to converge on small data; more risks overfitting |
| `warmup_steps` | 50 | Ramp LR up slowly at the start so early updates don't destabilize weights |
| `weight_decay` | 0.01 | L2 regularization — penalizes large weights, reduces overfitting |
| `batch_size` | 16 | Fits in memory; larger batches can hurt generalization on small datasets |
| `class_weight` | inverse frequency | Counteracts the 7.6:1 dark:light imbalance |

**Why class weights matter here:**
Without them, the cross-entropy loss is dominated by the dark class (87% of data).
The model learns to predict dark for everything and achieves 87% accuracy while
completely failing on light kinases. We weight the loss inversely: light samples
count ~7.6x more than dark samples during training.

> **🎙️ Interview flag:** *What is fine-tuning vs training from scratch?*
> Fine-tuning starts from pre-trained weights that already encode biomedical language.
> We adapt those weights slightly for our specific task — much more efficient than
> training a 110M parameter model from random weights on 1,200 samples.

In [ ]:
def tokenize(examples):
    return tokenizer(
        examples["ptm_text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

# Convert splits to HuggingFace Dataset format (required by Trainer API)
hf_train = Dataset.from_pandas(train_df.rename(columns={"label": "labels"}).reset_index(drop=True))
hf_val   = Dataset.from_pandas(val_df.rename(columns={"label": "labels"}).reset_index(drop=True))
hf_test  = Dataset.from_pandas(test_df.rename(columns={"label": "labels"}).reset_index(drop=True))

hf_train = hf_train.map(tokenize, batched=True)
hf_val   = hf_val.map(tokenize, batched=True)
hf_test  = hf_test.map(tokenize, batched=True)

for ds in [hf_train, hf_val, hf_test]:
    ds.set_format("torch")

print("Tokenization complete.")

In [ ]:
# Compute inverse-frequency class weights
# Formula: weight_i = N_total / (N_classes * N_i)
# This makes each class contribute equally to the loss regardless of how many samples it has
n_total = len(train_df)
n_classes = 2
class_counts = train_df["label"].value_counts().sort_index()

weights = [n_total / (n_classes * class_counts[i]) for i in range(n_classes)]
class_weights = torch.tensor(weights, dtype=torch.float)

for i, name in enumerate(CLASS_NAMES):
    print(f"  {name}: count={class_counts[i]}, weight={weights[i]:.3f}")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy":   accuracy_score(labels, preds),
        "f1_macro":   f1_score(labels, preds, average="macro"),
        "f1_light":   f1_score(labels, preds, pos_label=0, average="binary"),
    }


class WeightedTrainer(Trainer):
    """Trainer subclass that applies class weights to the cross-entropy loss."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = CrossEntropyLoss(weight=class_weights.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
# Load a fresh PubMedBERT with a 2-class head (weights randomly initialized)
ft_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
ft_model = ft_model.to(device)

training_args = TrainingArguments(
    output_dir="./ft_binary_model",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,          # ← critical: low LR preserves pre-trained knowledge
    warmup_steps=50,             # ← ramp LR up over first 50 steps
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",  # ← use F1-macro, not accuracy (imbalanced data!)
    greater_is_better=True,
    logging_steps=20,
)

trainer = WeightedTrainer(
    model=ft_model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning complete.")

In [ ]:
# Evaluate fine-tuned model on held-out test set
ft_preds_output = trainer.predict(hf_test)
ft_preds = np.argmax(ft_preds_output.predictions, axis=1)

print("=== Model 2: Fine-tuned PubMedBERT — Test Set ===")
print(classification_report(y_test, ft_preds, target_names=CLASS_NAMES))

cm2 = confusion_matrix(y_test, ft_preds)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm2, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(CLASS_NAMES); ax.set_yticklabels(CLASS_NAMES)
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm2[i, j], ha="center", va="center", fontsize=14)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Model 2: Fine-tuned — Confusion Matrix")
plt.tight_layout()
plt.show()

finetuned_results = {
    "model": "2. Fine-tuned PubMedBERT",
    "accuracy":  accuracy_score(y_test, ft_preds),
    "f1_macro":  f1_score(y_test, ft_preds, average="macro"),
    "precision": precision_score(y_test, ft_preds, average="macro"),
    "recall":    recall_score(y_test, ft_preds, average="macro"),
}

---
## Model 3: Zero-shot GPT-4.1

**What zero-shot means:** We give the model a task description and ask it to
classify — no training examples, no fine-tuning. The model uses only what it
already knows from pre-training on vast scientific literature.

**Why this is interesting to compare:**
- If zero-shot GPT-4.1 beats fine-tuned PubMedBERT, it suggests our training
  dataset is too small or noisy to improve on a powerful prior.
- If fine-tuned wins, it means task-specific training on our labels adds real signal.

**How the prompt works:**
The prompt tells GPT-4.1 to use *text richness and specificity* as the signal —
which is the honest proxy we have. We also explicitly warn it about the CDK2 confound:
a dark kinase can appear as the substrate of a well-known kinase (e.g. CDK2 phosphorylates
a dark prey), so the presence of a famous kinase name doesn't mean the protein is light.

> **🎙️ Interview flag:** *What is zero-shot prompting?*
> You describe the task in natural language without showing any examples.
> The model maps your description onto patterns it learned during pre-training.
> Contrast with *few-shot* prompting, where you include 2–5 labeled examples
> in the prompt to anchor the model's behavior.

> **🎙️ Interview flag:** *Why might zero-shot beat fine-tuning here?*
> GPT-4.1 has read most of PubMed. It may already "know" which kinases are
> well-studied and which aren't — giving it an advantage our 1,200-sample
> training set can't match. Small fine-tuning datasets can actually *hurt*
> a strong model by overfitting to label noise.

**Note:** We run zero-shot only on the **test set** (~122 samples) to limit API calls.

In [ ]:
ZERO_SHOT_PROMPT = """\
You are a bioinformatics expert classifying proteins based on their PTM \
(post-translational modification) annotation from UniProt.

Classify the protein as either:
- "light": well-studied kinase with detailed phosphorylation annotations — \
named upstream kinases, specific residue positions (e.g. Ser-473), \
and multiple literature references.
- "dark": understudied kinase with sparse or vague annotations — generic \
phrases like "phosphorylated", "by similarity", or non-phosphorylation \
modifications only.

IMPORTANT: A dark kinase may appear as the substrate of a well-known kinase \
(e.g. "Phosphorylated by CDK2"). This does NOT make it a light kinase. \
Focus on how specifically the annotation describes this protein's OWN \
phosphorylation biology.

PTM annotation:
\"{ptm_text}\"

Respond with exactly one word: light or dark"""


def zero_shot_predict(ptm_text: str, client) -> int:
    """Call GPT-4.1 zero-shot and return 0 (light) or 1 (dark).
    
    Returns 1 (dark) on any parse failure — conservative default since
    dark is the majority class and API errors shouldn't flip predictions.
    """
    prompt = ZERO_SHOT_PROMPT.format(ptm_text=ptm_text)
    try:
        response = client.invoke(prompt)
        answer = response.content.strip().lower()
        if "light" in answer:
            return 0
        return 1  # 'dark' or anything unexpected → dark
    except Exception:
        return 1


# Show the prompt so you can see exactly what GPT-4.1 receives
sample_text = test_df["ptm_text"].iloc[0]
print("=== Example prompt sent to GPT-4.1 ===")
print(ZERO_SHOT_PROMPT.format(ptm_text=sample_text[:80] + "..."))

In [ ]:
# Run zero-shot on test set — this makes one API call per sample
llm_client = get_llm_client()

test_texts = test_df["ptm_text"].tolist()
zs_preds = []

for i, text in enumerate(test_texts):
    pred = zero_shot_predict(text, llm_client)
    zs_preds.append(pred)
    if (i + 1) % 20 == 0:
        print(f"  {i + 1}/{len(test_texts)} done...")

zs_preds = np.array(zs_preds)
print(f"\nDone. Predicted light: {(zs_preds == 0).sum()} | dark: {(zs_preds == 1).sum()}")

In [ ]:
print("=== Model 3: Zero-shot GPT-4.1 — Test Set ===")
print(classification_report(y_test, zs_preds, target_names=CLASS_NAMES))

cm3 = confusion_matrix(y_test, zs_preds)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm3, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(CLASS_NAMES); ax.set_yticklabels(CLASS_NAMES)
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm3[i, j], ha="center", va="center", fontsize=14)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Model 3: Zero-shot GPT-4.1 — Confusion Matrix")
plt.tight_layout()
plt.show()

zeroshot_results = {
    "model": "3. Zero-shot GPT-4.1",
    "accuracy":  accuracy_score(y_test, zs_preds),
    "f1_macro":  f1_score(y_test, zs_preds, average="macro"),
    "precision": precision_score(y_test, zs_preds, average="macro"),
    "recall":    recall_score(y_test, zs_preds, average="macro"),
}

---
## Comparison

Primary metric: **F1-macro** — the unweighted average of F1 for each class.
This treats both light and dark kinase prediction as equally important,
regardless of how imbalanced the dataset is.

Accuracy alone is misleading here — a model predicting *dark* for every sample
would score 87% accuracy with 0% recall on the minority (light) class.

In [ ]:
results_df = pd.DataFrame([baseline_results, finetuned_results, zeroshot_results])
results_df = results_df.set_index("model")

# Format as percentages for readability
display_df = (results_df * 100).round(1).astype(str) + "%"
print("=== Model Comparison (Test Set) ===")
print(display_df.to_string())

# Bar chart of F1-macro across models
fig, ax = plt.subplots(figsize=(8, 4))
models = [r.split(".", 1)[1].strip() for r in results_df.index]
f1_scores = results_df["f1_macro"].values
bars = ax.bar(models, f1_scores, color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_ylim(0, 1)
ax.set_ylabel("F1-macro")
ax.set_title("Model Comparison — F1-macro on Test Set")
for bar, score in zip(bars, f1_scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{score:.3f}", ha="center", va="bottom", fontsize=11)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()